# BO Manual Evaluation Overview (Week 12)

This notebook aggregates manually evaluated weekly points from all function notebooks (`week_12_function_1..8_analysis.ipynb`), computes running best performance per function, and highlights evaluations that achieved a new maximum.

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

base_dir = Path.cwd()
week12_dir = base_dir / "notebooks" / "week_12"
files = sorted(week12_dir.glob("week_12_function_*_analysis.ipynb"))

pattern_x = re.compile(r"X_new_point_week_(\d+)\s*=\s*np\.array\(\s*\[\s*\[(.*?)\]\s*\]\s*\)", re.S)
pattern_y = re.compile(r"y_new_point_week_(\d+)\s*=\s*np\.array\(\s*\[(.*?)\]\s*\)", re.S)


def parse_float_list(text):
    parts = [p.strip() for p in text.replace("\n", " ").split(",") if p.strip()]
    return [float(p) for p in parts]


rows = []
for fp in files:
    mfunc = re.search(r"function_(\d+)_analysis", fp.name)
    if not mfunc:
        continue
    fn = int(mfunc.group(1))

    nb = json.loads(fp.read_text(encoding="utf-8"))
    code_text = "\n".join(
        "".join(c.get("source", []))
        for c in nb.get("cells", [])
        if c.get("cell_type") == "code"
    )

    x_map, y_map = {}, {}
    for wk, payload in pattern_x.findall(code_text):
        x_map[int(wk)] = parse_float_list(payload)
    for wk, payload in pattern_y.findall(code_text):
        vals = parse_float_list(payload)
        if vals:
            y_map[int(wk)] = vals[0]

    common_weeks = sorted(set(x_map).intersection(y_map))
    running_max = None
    for wk in common_weeks:
        yv = float(y_map[wk])
        running_max = yv if running_max is None else max(running_max, yv)
        is_new_max = abs(yv - running_max) < 1e-12
        rows.append(
            {
                "Function": f"Function {fn}",
                "FunctionID": fn,
                "Week": wk,
                "Point": ", ".join(f"{v:.6f}" for v in x_map[wk]),
                "y": yv,
                "New_Max": is_new_max,
            }
        )

df = pd.DataFrame(rows).sort_values(["FunctionID", "Week"]).reset_index(drop=True)
if df.empty:
    raise RuntimeError("No manual weekly evaluation points were parsed. Check variable naming in week_12 notebooks.")

df["Cell"] = df.apply(lambda r: f"y={r['y']:.4f}\n({r['Point']})", axis=1)

matrix = df.pivot(index="Function", columns="Week", values="Cell")
mask_new = (
    df.pivot(index="Function", columns="Week", values="New_Max")
    .reindex_like(matrix)
    .fillna(False)
)


def highlight_new_max(_):
    styles = pd.DataFrame("", index=matrix.index, columns=matrix.columns)
    styles[mask_new] = "background-color: #c6efce; font-weight: 600;"
    return styles


print("Manual evaluation matrix (green = new max at that week)")
display(matrix.style.apply(highlight_new_max, axis=None).set_properties(**{"white-space": "pre-wrap"}))

summary = (
    df.groupby(["Function", "FunctionID"], as_index=False)
    .agg(
        Evaluations=("Week", "count"),
        New_Max_Count=("New_Max", "sum"),
        Best_y=("y", "max"),
    )
    .sort_values("FunctionID")
)
summary["New_Max_Rate_%"] = (100 * summary["New_Max_Count"] / summary["Evaluations"]).round(1)
summary = summary[["Function", "Evaluations", "New_Max_Count", "New_Max_Rate_%", "Best_y"]]

print("\nBO success overview by function (how often a weekly point set a new max):")
display(summary)

overall_rate = 100.0 * df["New_Max"].sum() / len(df)
print(f"Overall: {df['New_Max'].sum()}/{len(df)} weekly evaluations were new maxima ({overall_rate:.1f}%).")